**1.  Ingestão de dados (camada Bronze)**

In [0]:
from pyspark.sql.functions import col, lit

volume_path = "/Volumes/mvp/mvp_data_engineering/mvp_comex/"

# 1. Leitura das tabelas Fato com encoding adequado para caracteres especiais
df_exp_24 = spark.read.csv(volume_path + "EXP_2024.csv", sep=";", header=True, inferSchema=True, encoding="latin1")
df_exp_25 = spark.read.csv(volume_path + "EXP_2025.csv", sep=";", header=True, inferSchema=True, encoding="latin1")
df_imp_24 = spark.read.csv(volume_path + "IMP_2024.csv", sep=";", header=True, inferSchema=True, encoding="latin1")
df_imp_25 = spark.read.csv(volume_path + "IMP_2025.csv", sep=";", header=True, inferSchema=True, encoding="latin1")

# Adição de coluna classificadora e união dos conjuntos anuais
df_exp = df_exp_24.unionByName(df_exp_25, allowMissingColumns=True).withColumn("TIPO_OPERACAO", lit("EXP"))
df_imp = df_imp_24.unionByName(df_imp_25, allowMissingColumns=True).withColumn("TIPO_OPERACAO", lit("IMP"))

# União dos fluxos de Exportação e Importação em estrutura única
df_bronze_fato = df_exp.unionByName(df_imp, allowMissingColumns=True)

# Persistência da tabela Fato na camada Bronze
df_bronze_fato.write.format("delta").mode("overwrite").saveAsTable("mvp.mvp_data_engineering.bronze_fato_comex")

# 2. Leitura e persistência das tabelas de Dimensão com encoding adequado
dimensoes = ["NCM", "PAIS", "UF", "VIA"]
for dim in dimensoes:
    df_dim = spark.read.csv(f"{volume_path}{dim}.csv", sep=";", header=True, inferSchema=True, encoding="latin1")
    df_dim.write.format("delta").mode("overwrite").saveAsTable(f"mvp.mvp_data_engineering.bronze_dim_{dim.lower()}")

**2. Limpeza e padronização (camada Silver)**

In [0]:
from pyspark.sql.functions import col, when

# Leitura da tabela Fato a partir da camada Bronze
df_silver_fato = spark.read.table("mvp.mvp_data_engineering.bronze_fato_comex")

# Tratamento de Qualidade de Dados
# 1. Padronização de valores ausentes na identificação de UF
df_silver_fato = df_silver_fato.withColumn("SG_UF_NCM", 
                                           when(col("SG_UF_NCM") == "ND", "Nao Informado")
                                           .otherwise(col("SG_UF_NCM")))

# 2. Conversão de tipos de dados financeiros e de peso para Double
df_silver_fato = df_silver_fato.withColumn("VL_FOB", col("VL_FOB").cast("double")) \
                               .withColumn("KG_LIQUIDO", col("KG_LIQUIDO").cast("double"))

# 3. Filtragem de registros com valores financeiros inválidos ou zerados
df_silver_fato = df_silver_fato.filter(col("VL_FOB") > 0)

# Persistência da tabela tratada na camada Silver
df_silver_fato.write.format("delta").mode("overwrite").saveAsTable("mvp.mvp_data_engineering.silver_fato_comex")

**3. Tabela final (camada Gold)**

In [0]:
from pyspark.sql.functions import col, coalesce, lit

# 1. Leitura da tabela Silver e das Dimensões
df_fato = spark.read.table("mvp.mvp_data_engineering.silver_fato_comex")
df_dim_ncm = spark.read.table("mvp.mvp_data_engineering.bronze_dim_ncm")
df_dim_pais = spark.read.table("mvp.mvp_data_engineering.bronze_dim_pais")
df_dim_uf = spark.read.table("mvp.mvp_data_engineering.bronze_dim_uf")
df_dim_via = spark.read.table("mvp.mvp_data_engineering.bronze_dim_via")

# 2. Cruzamento de dadoscom conversão para String nas chaves
df_gold = df_fato.join(df_dim_ncm, df_fato.CO_NCM.cast("string") == df_dim_ncm.CO_NCM.cast("string"), "left") \
                 .join(df_dim_pais, df_fato.CO_PAIS.cast("string") == df_dim_pais.CO_PAIS.cast("string"), "left") \
                 .join(df_dim_uf, df_fato.SG_UF_NCM.cast("string") == df_dim_uf.SG_UF.cast("string"), "left") \
                 .join(df_dim_via, df_fato.CO_VIA.cast("string") == df_dim_via.CO_VIA.cast("string"), "left")

# 3. Seleção e renomeação com tratamento de nulos via coalesce garantindo tipos iguais
df_gold_final = df_gold.select(
    df_fato.CO_ANO.alias("ano"),
    df_fato.CO_MES.alias("mes"),
    df_fato.TIPO_OPERACAO.alias("tipo_operacao"),
    df_fato.CO_NCM.alias("cod_ncm"),
    coalesce(df_dim_ncm.NO_NCM_POR, lit("Produto Não Informado")).alias("desc_produto"),
    coalesce(df_dim_pais.NO_PAIS, lit("País Não Informado")).alias("pais"),
    coalesce(df_dim_uf.NO_UF, lit("Estado Não Informado")).alias("estado_uf"),
    coalesce(df_dim_via.NO_VIA, lit("Via Não Informada")).alias("via_transporte"),
    df_fato.KG_LIQUIDO.alias("peso_liquido_kg"),
    df_fato.VL_FOB.alias("valor_fob_dolar")
)

# 4. Persistência da tabela analítica na camada Gold 
df_gold_final.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("mvp.mvp_data_engineering.gold_comex_analitica")

**4. Qualidade de dados**

In [0]:
from pyspark.sql.functions import col, count, when, isnull

# Leitura da tabela final (Gold)
df_auditoria = spark.read.table("mvp.mvp_data_engineering.gold_comex_analitica")

# 1. Verificação de integridade referencial e valores nulos
df_qualidade_nulos = df_auditoria.select(
    count(when(isnull("cod_ncm"), 1)).alias("nulos_ncm"),
    count(when(isnull("pais"), 1)).alias("nulos_pais"),
    count(when(isnull("estado_uf"), 1)).alias("nulos_uf"),
    count(when(isnull("valor_fob_dolar"), 1)).alias("nulos_valor")
)

print("--- Relatório de Nulos ---")
display(df_qualidade_nulos)

# 2. Verificação de regras de negócio (Valores financeiros e físicos válidos)
total_registros = df_auditoria.count()
registros_invalidos_fob = df_auditoria.filter(col("valor_fob_dolar") <= 0).count()
registros_invalidos_peso = df_auditoria.filter(col("peso_liquido_kg") < 0).count()

print("\n--- Validação de Regras de Negócio ---")
print(f"Total de registros analisados: {total_registros}")
print(f"Registros com Valor FOB inválido (<= 0): {registros_invalidos_fob}")
print(f"Registros com Peso Líquido inválido (< 0): {registros_invalidos_peso}")

--- Relatório de Nulos ---


nulos_ncm,nulos_pais,nulos_uf,nulos_valor
0,0,0,0



--- Validação de Regras de Negócio ---
Total de registros analisados: 7969356
Registros com Valor FOB inválido (<= 0): 0
Registros com Peso Líquido inválido (< 0): 0


**5. Análise de dados**

**5.1. Vulnerabilidade da Cadeia de Suprimentos**

In [0]:
%sql
SELECT 
    CASE 
        WHEN LOWER(desc_produto) LIKE '%fertilizante%' OR LOWER(desc_produto) LIKE '%adubo%' OR LOWER(desc_produto) LIKE '%inseticida%' OR LOWER(desc_produto) LIKE '%herbicida%' THEN 'Insumos Agrícolas'
        WHEN LOWER(desc_produto) LIKE '%semicondutor%' OR LOWER(desc_produto) LIKE '%circuito integrado%' OR LOWER(desc_produto) LIKE '%processador%' OR LOWER(desc_produto) LIKE '%diodo%' THEN 'Tecnologia'
        WHEN LOWER(desc_produto) LIKE '%hulha%' OR LOWER(desc_produto) LIKE '%carvão mineral%' OR LOWER(desc_produto) LIKE '%gás natural%' OR LOWER(desc_produto) LIKE '%petróleo%' THEN 'Minérios e Energia'
        WHEN LOWER(desc_produto) LIKE '%farmacêutic%' OR LOWER(desc_produto) LIKE '%medicamento%' OR LOWER(desc_produto) LIKE '%vacina%' OR LOWER(desc_produto) LIKE '%médic%' OR LOWER(desc_produto) LIKE '%hospitalar%' THEN 'Fármacos e Equipamentos Médicos'
        WHEN LOWER(desc_produto) LIKE '%polímero%' OR LOWER(desc_produto) LIKE '%resina%' OR LOWER(desc_produto) LIKE '%catalisador%' OR LOWER(desc_produto) LIKE '%composto químico%' THEN 'Química Fina e Polímeros'
        ELSE 'Outros'
    END AS categoria_critica,
    ROUND(SUM(valor_fob_dolar) / 1000000000, 2) AS bilhoes_us,
    ROUND(SUM(peso_liquido_kg) / 1000000000, 2) AS milhoes_ton
FROM mvp.mvp_data_engineering.gold_comex_analitica
WHERE tipo_operacao = 'IMP'
  AND (
      LOWER(desc_produto) LIKE '%fertilizante%' OR LOWER(desc_produto) LIKE '%adubo%' OR LOWER(desc_produto) LIKE '%inseticida%' OR LOWER(desc_produto) LIKE '%herbicida%'
   OR LOWER(desc_produto) LIKE '%semicondutor%' OR LOWER(desc_produto) LIKE '%circuito integrado%' OR LOWER(desc_produto) LIKE '%processador%' OR LOWER(desc_produto) LIKE '%diodo%'
   OR LOWER(desc_produto) LIKE '%hulha%' OR LOWER(desc_produto) LIKE '%carvão mineral%' OR LOWER(desc_produto) LIKE '%gás natural%' OR LOWER(desc_produto) LIKE '%petróleo%'
   OR LOWER(desc_produto) LIKE '%farmacêutic%' OR LOWER(desc_produto) LIKE '%medicamento%' OR LOWER(desc_produto) LIKE '%vacina%' OR LOWER(desc_produto) LIKE '%médic%' OR LOWER(desc_produto) LIKE '%hospitalar%'
   OR LOWER(desc_produto) LIKE '%polímero%' OR LOWER(desc_produto) LIKE '%resina%' OR LOWER(desc_produto) LIKE '%catalisador%' OR LOWER(desc_produto) LIKE '%composto químico%'
  )
GROUP BY categoria_critica
ORDER BY bilhoes_us DESC;

categoria_critica,bilhoes_us,milhoes_ton
Minérios e Energia,29.65,84.9
Fármacos e Equipamentos Médicos,17.46,0.13
Insumos Agrícolas,11.88,13.71
Química Fina e Polímeros,8.23,3.6
Tecnologia,6.93,0.22


Databricks visualization. Run in Databricks to view.

**5.1.1. Vunerabilidade na cadeia de suprimentos por parceiro comercial**

In [0]:
%sql
WITH Base_Critica AS (
    SELECT 
        pais,
        valor_fob_dolar,
        peso_liquido_kg
    FROM mvp.mvp_data_engineering.gold_comex_analitica
    WHERE tipo_operacao = 'IMP'
      AND (
          LOWER(desc_produto) LIKE '%fertilizante%' OR LOWER(desc_produto) LIKE '%adubo%' OR LOWER(desc_produto) LIKE '%inseticida%' OR LOWER(desc_produto) LIKE '%herbicida%'
       OR LOWER(desc_produto) LIKE '%semicondutor%' OR LOWER(desc_produto) LIKE '%circuito integrado%' OR LOWER(desc_produto) LIKE '%processador%' OR LOWER(desc_produto) LIKE '%diodo%'
       OR LOWER(desc_produto) LIKE '%hulha%' OR LOWER(desc_produto) LIKE '%carvão mineral%' OR LOWER(desc_produto) LIKE '%gás natural%' OR LOWER(desc_produto) LIKE '%petróleo%'
       OR LOWER(desc_produto) LIKE '%farmacêutic%' OR LOWER(desc_produto) LIKE '%medicamento%' OR LOWER(desc_produto) LIKE '%vacina%' OR LOWER(desc_produto) LIKE '%médic%' OR LOWER(desc_produto) LIKE '%hospitalar%'
       OR LOWER(desc_produto) LIKE '%polímero%' OR LOWER(desc_produto) LIKE '%resina%' OR LOWER(desc_produto) LIKE '%catalisador%' OR LOWER(desc_produto) LIKE '%composto químico%'
      )
)
SELECT 
    pais AS pais_origem,
    ROUND(SUM(valor_fob_dolar) / 1000000000, 2) AS bilhoes_us,
    ROUND(SUM(peso_liquido_kg) / 1000000000, 2) AS milhoes_ton,
    ROUND((SUM(valor_fob_dolar) / (SELECT SUM(valor_fob_dolar) FROM Base_Critica)) * 100, 2) AS percentual_participacao
FROM Base_Critica
GROUP BY pais
ORDER BY bilhoes_us DESC
LIMIT 5;

pais_origem,bilhoes_us,milhoes_ton,percentual_participacao
Estados Unidos,17.83,35.15,24.05
China,10.81,6.99,14.58
Arábia Saudita,3.91,6.8,5.28
Alemanha,3.22,0.41,4.35
Bolívia,2.25,7.5,3.04


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**5.2. Transição energética e Tecnológica**

In [0]:
%sql
WITH Base_Transicao AS (
    SELECT 
        estado_uf,
        valor_fob_dolar
    FROM mvp.mvp_data_engineering.gold_comex_analitica
    WHERE tipo_operacao = 'IMP'
      AND (
          LOWER(desc_produto) LIKE '%fotovoltaic%'
       OR LOWER(desc_produto) LIKE '%lítio%'
       OR LOWER(desc_produto) LIKE '%aerogerador%'
       OR LOWER(desc_produto) LIKE '%eólic%'
       OR LOWER(desc_produto) LIKE '%semicondutor%'
       OR LOWER(desc_produto) LIKE '%circuito integrado%'
       OR LOWER(desc_produto) LIKE '%processador%'
       OR (LOWER(desc_produto) LIKE '%veículo%' AND LOWER(desc_produto) LIKE '%elétrico%')
       OR (LOWER(desc_produto) LIKE '%automóvel%' AND LOWER(desc_produto) LIKE '%elétrico%')
       OR LOWER(desc_produto) LIKE '%híbrido%'
      )
)
SELECT 
    estado_uf,
    ROUND(SUM(valor_fob_dolar) / 1000000000, 2) AS bilhoes_us,
    ROUND((SUM(valor_fob_dolar) / (SELECT SUM(valor_fob_dolar) FROM Base_Transicao)) * 100, 2) AS percentual_participacao
FROM Base_Transicao
GROUP BY estado_uf
ORDER BY bilhoes_us DESC
LIMIT 5;

estado_uf,bilhoes_us,percentual_participacao
Espírito Santo,6.79,34.52
Amazonas,3.21,16.34
São Paulo,2.86,14.54
Santa Catarina,2.42,12.31
Minas Gerais,0.8,4.07


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

**5.3. Qualidade da Balança Comercial**

In [0]:
%sql
WITH Top_Parceiros AS (
    -- 1. Isolar os 10 maiores parceiros comerciais pelo fluxo total (Exportação + Importação)
    SELECT 
        pais,
        SUM(valor_fob_dolar) AS fluxo_total_us
    FROM mvp.mvp_data_engineering.gold_comex_analitica
    WHERE pais IS NOT NULL AND pais != 'País Não Informado'
    GROUP BY pais
    ORDER BY fluxo_total_us DESC
    LIMIT 10
),
Balanca_Detalhada AS (
    -- 2. Calcular montantes e classificar a exportação de produtos primários (Commodities)
    SELECT 
        g.pais,
        p.fluxo_total_us,
        SUM(CASE WHEN g.tipo_operacao = 'EXP' THEN g.valor_fob_dolar ELSE 0 END) AS total_exportado,
        SUM(CASE WHEN g.tipo_operacao = 'IMP' THEN g.valor_fob_dolar ELSE 0 END) AS total_importado,
        SUM(CASE 
            WHEN g.tipo_operacao = 'EXP' AND (
                LOWER(g.desc_produto) LIKE '%soja%' 
                OR LOWER(g.desc_produto) LIKE '%minério%' 
                OR LOWER(g.desc_produto) LIKE '%petróleo%' 
                OR LOWER(g.desc_produto) LIKE '%carne%' 
                OR LOWER(g.desc_produto) LIKE '%açúcar%' 
                OR LOWER(g.desc_produto) LIKE '%café%' 
                OR LOWER(g.desc_produto) LIKE '%milho%' 
                OR LOWER(g.desc_produto) LIKE '%celulose%' 
                OR LOWER(g.desc_produto) LIKE '%algodão%' 
                OR LOWER(g.desc_produto) LIKE '%farelo%'
                OR LOWER(g.desc_produto) LIKE '%ouro%'
            ) THEN g.valor_fob_dolar 
            ELSE 0 
        END) AS exportacao_commodities
    FROM mvp.mvp_data_engineering.gold_comex_analitica g
    INNER JOIN Top_Parceiros p ON g.pais = p.pais
    GROUP BY g.pais, p.fluxo_total_us
)
-- 3. Consolidar os indicadores financeiros finais ajustando a escala
SELECT 
    pais,
    ROUND(fluxo_total_us / 1000000000, 2) AS montante_bilhoes_us,
    ROUND((total_exportado - total_importado) / 1000000000, 2) AS saldo_comercial_bilhoes_us,
    ROUND((exportacao_commodities / NULLIF(total_exportado, 0)) * 100, 2) AS percentual_commodities_exportadas
FROM Balanca_Detalhada
ORDER BY fluxo_total_us DESC;

pais,montante_bilhoes_us,saldo_comercial_bilhoes_us,percentual_commodities_exportadas
China,328.86,59.76,80.66
Estados Unidos,163.85,-7.74,21.02
Argentina,58.39,5.37,5.42
Alemanha,40.56,-15.82,30.69
Países Baixos (Holanda),28.12,18.77,54.0
México,27.51,3.55,13.08
Índia,27.33,-3.05,74.85
Espanha,26.55,10.96,78.55
Chile,23.41,4.25,32.15
Rússia,23.35,-17.4,25.06


Databricks visualization. Run in Databricks to view.